# ¿Es λ física o del telescopio? Puentes ER frente al difuminado instrumental

**Proyecto:** *Vacuum Particle Journey*, quinta parte.

**Lo que dijo el notebook 4:** la RAR con la masa extra repartida a escala λ ≈ 0.5 kpc mejora el ajuste de forma grande y robusta (ΔBIC ≈ +900, sin ninguna réplica bootstrap con λ = 0). Pero **λ crece con el tamaño de la galaxia** (ρ = 0.51, p = 10⁻⁷), algo que una propiedad física de los puentes ER no debería hacer.

**Dos explicaciones en competencia:**
1. **Instrumental (*beam smearing*):** los radiotelescopios difuminan la curva de rotación a una escala **angular** fija (el haz, del orden de 10–30″ en los estudios de hidrógeno neutro). En kpc eso es λ = θ·D, y crece con la distancia. Las galaxias grandes de la muestra suelen estar más lejos. 0.45 kpc a ~10 Mpc son ~9″.
2. **Física (red ER):** λ es una escala real del entrelazamiento. Puede ser constante, o depender de la **densidad** de la galaxia: en galaxias difusas, con menos decoherencia, la red podría extenderse más.

**Modelos que compiten** (todos con g† global y Υ por galaxia):

| Modelo | λ en cada galaxia | Parámetros globales extra |
|---|---|---|
| RAR local | 0 | – |
| λ constante | λ | 1 |
| **Instrumental** | θ · D (θ en segundos de arco) | 1 |
| **Densidad** | λ₀ (Σ_b/Σ_med)^γ | 2 |
| Tamaño | λ₀ (r_max/10 kpc)^γ | 2 |
| **Instrumental + físico** | √((θ·D)² + λ_f²) | 2 |

El modelo combinado es la prueba clave: una vez descontado el efecto del haz, ¿queda una λ_f física distinta de cero?

**Además:**
- se repite todo **quitando los puntos interiores** (dentro de 2 haces), donde el difuminado es mayor;
- se compara con NFW en las mismas condiciones (con el 5 % sistemático y con el corte).

In [ ]:
import os, io, re, glob, zipfile, urllib.request, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares, minimize_scalar
from scipy.special import i0, i1, k0, k1
from scipy.stats import spearmanr

warnings.filterwarnings("ignore", category=RuntimeWarning)
RNG = np.random.default_rng(42)
QUICK = False

COL = {"Datos": "#222222", "Solo bariones": "#8a8a85", "NFW ΛCDM (c–M)": "#6250d6",
       "RAR (g† = 1.2e-10)": "#e87ba4", "RAR local (g† global)": "#eb6834", "RAR no local (ER)": "#1baf7a", "Instrumental": "#2a78d6", "Densidad": "#eda100", "Tamaño": "#e34948"}
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.edgecolor": "#c3c2b7", "lines.linewidth": 2, "font.size": 10, "legend.frameon": False,
})

G = 4.30091e-6
C_KMS = 299792.458
H0 = 70.0; H0_KPC = H0 / 1000.0
KPC_M = 3.0857e19
TO_MS2 = 1e6 / KPC_M                    # (km/s)^2/kpc -> m/s^2

## 1. Datos SPARC

In [ ]:
DATA_DIR = "sparc_rotmod"
BASES = ["https://astroweb.case.edu/SPARC/", "http://astroweb.cwru.edu/SPARC/"]
ZENODO = "https://zenodo.org/records/16284118/files/Rotmod_LTG.zip?download=1"

def _get(url):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    return urllib.request.urlopen(req, timeout=60).read()

def _files():
    return sorted(glob.glob(os.path.join(DATA_DIR, "**", "*_rotmod.dat"), recursive=True))

def get_sparc():
    os.makedirs(DATA_DIR, exist_ok=True)
    if _files():
        return True
    for url in [b + "Rotmod_LTG.zip" for b in BASES] + [ZENODO]:
        try:
            print("Descargando", url)
            zipfile.ZipFile(io.BytesIO(_get(url))).extractall(DATA_DIR)
            if _files():
                return True
        except Exception as e:
            print("  falló:", type(e).__name__)
    try:
        from google.colab import files
        print("Sube Rotmod_LTG.zip (de astroweb.case.edu/SPARC)")
        for name, content in files.upload().items():
            zipfile.ZipFile(io.BytesIO(content)).extractall(DATA_DIR)
        return bool(_files())
    except Exception:
        return False

def get_quality():
    for b in BASES:
        try:
            txt = _get(b + "SPARC_Lelli2016c.mrt").decode("latin-1")
            q = {}
            for line in txt.splitlines():
                t = line.split()
                if len(t) >= 18 and re.fullmatch(r"\d+", t[1] or "") and t[17] in ("1", "2", "3"):
                    q[t[0]] = dict(Q=int(t[17]), Inc=float(t[5]))
            if len(q) > 100:
                return q
        except Exception:
            pass
    return None

def load_rotmod(path):
    name = os.path.basename(path).replace("_rotmod.dat", "")
    d = np.atleast_2d(np.loadtxt(path, comments="#"))
    g = dict(name=name, r=d[:, 0], vobs=d[:, 1], err=d[:, 2], vgas=d[:, 3], vdisk=d[:, 4], vbul=d[:, 5])
    ok = (g["r"] > 0) & (g["err"] > 0)
    o = np.argsort(g["r"][ok])
    for k in ["r", "vobs", "err", "vgas", "vdisk", "vbul"]:
        g[k] = g[k][ok][o]
    g["vgas2"] = g["vgas"] * np.abs(g["vgas"])
    g["vstar2"] = g["vdisk"] * np.abs(g["vdisk"]) + 1.4 * g["vbul"] * np.abs(g["vbul"])
    return g

def _v2_expdisk(r, M, Rd):
    y = r / (2 * Rd)
    return 4 * np.pi * G * (M / (2 * np.pi * Rd**2)) * Rd * y**2 * (i0(y) * k0(y) - i1(y) * k1(y))

def _v2_nfw(r, V200, c):
    x = r / (V200 / (10 * H0_KPC))
    mu = lambda y: np.log1p(y) - y / (1 + y)
    return V200**2 * mu(c * x) / (x * mu(c))

def make_synthetic_set(n=30, seed=0):
    rng = np.random.default_rng(seed)
    out = {}
    for name in ["NGC3198", "NGC2403", "DDO154"] + [f"SYN{i:03d}" for i in range(n - 3)]:
        V200 = 10 ** rng.uniform(1.6, 2.4); c = 10 ** rng.normal(1.0, 0.12)
        Rd = rng.uniform(1, 5); Ml = 10 ** rng.uniform(8.5, 11); Mg = Ml * 10 ** rng.uniform(-1, 0.7)
        r = np.linspace(0.3 * Rd, rng.uniform(5, 9) * Rd, rng.integers(12, 40))
        vdisk = np.sqrt(_v2_expdisk(r, Ml, Rd)); vgas = np.sqrt(_v2_expdisk(r, Mg, 2.5 * Rd))
        vtrue = np.sqrt(vgas**2 + 0.5 * vdisk**2 + _v2_nfw(r, V200, c))
        err = 2 + 0.04 * vtrue
        os.makedirs("synthetic", exist_ok=True)
        path = os.path.join("synthetic", f"{name}_rotmod.dat")
        np.savetxt(path, np.c_[r, vtrue + rng.normal(0, err), err, vgas, vdisk, 0 * r], header="Distance = 10 Mpc")
        out[name] = load_rotmod(path)
    return out

HAVE_SPARC = get_sparc()
if HAVE_SPARC:
    GAL = {}
    for f in _files():
        g = load_rotmod(f)
        if len(g["r"]) >= 5:
            GAL[g["name"]] = g
    Q = get_quality()
    if Q:
        GAL = {k: v for k, v in GAL.items() if k in Q and Q[k]["Q"] < 3 and Q[k]["Inc"] >= 30}
        print("Corte de calidad aplicado (Q<3, inc≥30°).")
    else:
        print("Sin tabla de calidad: se usan todas las galaxias.")
else:
    print("⚠️  SIN DATOS REALES: galaxias SINTÉTICAS con halo NFW. Los resultados no tienen valor físico.")
    GAL = make_synthetic_set()

if QUICK and len(GAL) > 40:
    keep = [k for k in ["NGC3198", "NGC2403", "DDO154"] if k in GAL]
    rest = [k for k in GAL if k not in keep]
    GAL = {k: GAL[k] for k in keep + list(RNG.choice(rest, 40 - len(keep), replace=False))}
N_TOT = sum(len(g["r"]) for g in GAL.values())
print(f"Galaxias: {len(GAL)} | puntos: {N_TOT}")

## 2. Modelo RAR no local (del notebook 4)

In [ ]:
NG = 400
U_GRID = np.geomspace(0.1, 1.5, 40)
LAM_GRID = np.r_[0.0, np.geomspace(0.1, 100, 24)]                                       # kpc
GD_GRID = np.unique(np.r_[np.geomspace(0.6e-10, 2.4e-10, 25), 1.2e-10]) / TO_MS2        # (km/s)^2/kpc
I_GD12 = int(np.argmin(np.abs(GD_GRID * TO_MS2 - 1.2e-10)))

def vb2(g, u):
    return g["vgas2"] + u * g["vstar2"]

def prior(u):
    return np.log10(u / 0.5) / 0.1

def rar_g(gb, gd):
    return gb / (-np.expm1(-np.sqrt(gb / gd)))

def F_matrix(r, a, lam):
    R, A = r[:, None], a[None, :]
    if lam == 0:
        return (A < R).astype(float)
    x = R / lam
    e_lo = np.exp(-np.clip((A - R) / lam, 0, None))
    e_hi = np.exp(-np.clip((R - A) / lam, 0, None))
    e_sum = np.exp(-(A + R) / lam)
    I_lo = e_lo * (x - 1) + e_sum * (1 + x)
    I_hi = 2 * A / lam - e_hi * (1 + x) + e_sum * (1 + x)
    return np.where(R <= A, I_lo, I_hi) * lam / (2 * A)

def setup(g):
    if "_s" in g:
        return g["_s"]
    rmax = g["r"].max()
    rg = np.linspace(rmax / NG, rmax, NG)
    s = dict(rg=rg, mids=0.5 * (np.r_[0.0, rg[:-1]] + rg),
             vg=np.interp(rg, np.r_[0.0, g["r"]], np.r_[0.0, g["vgas2"]]),
             vs=np.interp(rg, np.r_[0.0, g["r"]], np.r_[0.0, g["vstar2"]]))
    g["_s"] = s
    return s

def phantom_shells(s, U, GD):
    # G·M_fantasma en cascarones -> forma (nΥ, ng†, NG)
    v2 = s["vg"][None, :] + U[:, None] * s["vs"][None, :]
    gb = np.clip(v2, 1e-6, None) / s["rg"][None, :]
    gr = rar_g(gb[:, None, :], GD[None, :, None])
    GM = s["rg"][None, None, :] ** 2 * (gr - gb[:, None, :])
    return np.diff(np.concatenate([np.zeros(GM.shape[:2] + (1,)), GM], axis=2), axis=2)

def model_v2(g, u, lam, gd):
    s = setup(g)
    d = phantom_shells(s, np.array([u]), np.array([gd]))[0, 0]
    return vb2(g, u) + F_matrix(g["r"], s["mids"], lam) @ d / g["r"]

# Comprobaciones: F(λ→0) es un escalón; con λ pequeño se recupera la RAR local exacta
_g = next(iter(GAL.values()))
v_loc = np.sqrt(_g["r"] * rar_g(np.clip(vb2(_g, 0.5), 1e-6, None) / _g["r"], 1.2e-10 / TO_MS2))
v_0 = np.sqrt(model_v2(_g, 0.5, 0.0, 1.2e-10 / TO_MS2))
v_s = np.sqrt(model_v2(_g, 0.5, 1e-3, 1.2e-10 / TO_MS2))
print(f"Test: |v(λ=0) − RAR exacta| máx = {np.max(np.abs(v_0 - v_loc)):.2f} km/s; "
      f"|v(λ=0.001) − v(λ=0)| máx = {np.max(np.abs(v_s - v_0)):.3f} km/s (ambos deben ser ≈ 0)")

## 3. Distancia y densidad de cada galaxia

- **Distancia D** [Mpc]: viene en la cabecera de cada archivo de SPARC.
- **Densidad superficial bariónica** media dentro del último radio: $\Sigma_b = M_b(r_{max})/(\pi r_{max}^2)$, con Υ = 0.5, en M☉/pc².
- Conversión a ángulo: $\theta[″] = 206265 \cdot \lambda[\text{kpc}] / (1000\, D[\text{Mpc}])$.

In [ ]:
ARCSEC = 206265.0
paths = _files() if HAVE_SPARC else glob.glob(os.path.join("synthetic", "*_rotmod.dat"))
DIST = {}
for f in paths:
    with open(f) as fh:
        for line in fh:
            m = re.search(r"Distance\s*=\s*([\d.]+)", line)
            if m:
                DIST[os.path.basename(f).replace("_rotmod.dat", "")] = float(m.group(1)); break
if not HAVE_SPARC:
    _rd = np.random.default_rng(3)
    DIST = {k: float(10 ** _rd.uniform(0.5, 1.8)) for k in GAL}
    print("⚠️  Distancias sintéticas aleatorias (solo para probar el código)")
GAL = {k: g for k, g in GAL.items() if k in DIST}
NAMES = list(GAL)
D = np.array([DIST[k] for k in NAMES])
RMAX = np.array([GAL[k]["r"].max() for k in NAMES])
MB = np.array([max(GAL[k]["r"][-1] * vb2(GAL[k], 0.5)[-1] / G, 1e5) for k in NAMES])
SIGMA = MB / (np.pi * (RMAX * 1e3) ** 2)          # M☉/pc²
N_TOT_PTS = sum(len(g["r"]) for g in GAL.values())
print(f"{len(NAMES)} galaxias | D: {D.min():.1f}–{D.max():.1f} Mpc | Σ_b: {np.median(SIGMA):.1f} M☉/pc² (mediana)")
rho, p = spearmanr(np.log10(D), np.log10(RMAX))
print(f"Correlación distancia–tamaño en la muestra: ρ = {rho:.2f} (p = {p:.0e})  ← por esto λ instrumental imitaría 'λ ∝ tamaño'")

## 4. Tablas de χ²

Igual que en el notebook 4, pero con una rejilla de λ más fina (46 valores) y tres versiones:
1. errores originales;
2. con 5 % de error sistemático;
3. con 5 % sistemático **y sin los puntos dentro de 2 haces** (`BEAM_ARCSEC`, 15″ por defecto). El modelo usa la masa bariónica en todos los radios; solo se ocultan esas velocidades en el χ².

In [ ]:
LAM_GRID = np.r_[0.0, np.geomspace(0.02, 200, 45)]
LOGL = np.log(np.r_[0.01, LAM_GRID[1:]])
BEAM_ARCSEC = 15.0

def chi_table(floor=0.0, cut_beams=None):
    TOT = np.zeros((len(NAMES), len(LAM_GRID), len(GD_GRID))); DAT = np.zeros_like(TOT); NPT = np.zeros(len(NAMES), int)
    for n_i, name in enumerate(NAMES):
        g = GAL[name]; s = setup(g)
        err = np.sqrt(g["err"] ** 2 + (floor * g["vobs"]) ** 2)
        mask = np.ones(len(g["r"]), bool)
        if cut_beams:
            mask = g["r"] >= cut_beams * BEAM_ARCSEC / ARCSEC * D[n_i] * 1e3
            if mask.sum() < 5:
                mask = np.zeros(len(g["r"]), bool); mask[-5:] = True
        NPT[n_i] = mask.sum()
        d = phantom_shells(s, U_GRID, GD_GRID)
        base = vb2(g, U_GRID[:, None]); pr = prior(U_GRID) ** 2
        for l_i, lam in enumerate(LAM_GRID):
            extra = np.einsum("in,ugn->ugi", F_matrix(g["r"], s["mids"], lam), d) / g["r"]
            v = np.sqrt(np.clip(base[:, None, :] + extra, 0, None))
            c = ((((g["vobs"] - v) / err) ** 2) * mask).sum(-1)
            t = c + pr[:, None]; k = t.argmin(0); a = np.arange(len(GD_GRID))
            TOT[n_i, l_i] = t[k, a]; DAT[n_i, l_i] = c[k, a]
    return dict(tot=TOT, dat=DAT, npts=NPT)

TABLES = {}
for label, kw in [("Errores originales", {}), ("5 % sistemático", dict(floor=0.05)),
                  ("5 % + sin interior", dict(floor=0.05, cut_beams=2))]:
    t0 = time.time(); TABLES[label] = chi_table(**kw)
    print(f"{label}: {time.time()-t0:.0f} s, {TABLES[label]['npts'].sum()} puntos")

## 5. Competición de modelos para λ

Para cada modelo se recorre una rejilla de sus parámetros globales (y de g†), con Υ ya optimizado en cada galaxia. El BIC cuenta Υ por galaxia + g† + los parámetros de λ.

In [ ]:
def per_gal(tab, lam_vec, which="tot"):
    T = tab[which]; n = np.arange(T.shape[0])
    ll = np.log(np.clip(lam_vec, 0.01, 200))
    k = np.clip(np.searchsorted(LOGL, ll), 1, len(LOGL) - 1)
    w = np.clip((ll - LOGL[k - 1]) / (LOGL[k] - LOGL[k - 1]), 0, 1)[:, None]
    return T[n, k - 1, :] * (1 - w) + T[n, k, :] * w               # (galaxias, g†)

SIG_MED = np.median(SIGMA)
TH = lambda th: th / ARCSEC * D * 1e3                                # θ[″] -> λ[kpc] por galaxia
MODELS = {
    "RAR local":            (lambda: [dict()], lambda p: np.zeros(len(NAMES)), 0),
    "λ constante":          (lambda: [dict(L=L) for L in np.geomspace(0.02, 200, 70)], lambda p: np.full(len(NAMES), p["L"]), 1),
    "Instrumental (θ·D)":   (lambda: [dict(th=t) for t in np.geomspace(0.3, 300, 70)], lambda p: TH(p["th"]), 1),
    "Densidad λ(Σ)":        (lambda: [dict(L=L, gm=gm) for L in np.geomspace(0.02, 100, 40) for gm in np.linspace(-1.5, 1.5, 31)],
                             lambda p: p["L"] * (SIGMA / SIG_MED) ** p["gm"], 2),
    "Tamaño λ(r_max)":      (lambda: [dict(L=L, gm=gm) for L in np.geomspace(0.02, 100, 40) for gm in np.linspace(-1.5, 1.5, 31)],
                             lambda p: p["L"] * (RMAX / 10) ** p["gm"], 2),
    "Instrumental + físico": (lambda: [dict(th=t, Lf=Lf) for t in np.r_[0, np.geomspace(0.3, 300, 40)] for Lf in np.r_[0, np.geomspace(0.02, 50, 39)]],
                             lambda p: np.sqrt(TH(p["th"]) ** 2 + p["Lf"] ** 2), 2),
}

def compete(tab):
    rows, best = [], {}
    npts = tab["npts"].sum()
    for m, (grid, lamf, k) in MODELS.items():
        bt, bp, bj = np.inf, None, None
        for p in grid():
            S = per_gal(tab, lamf(p)).sum(0); j = int(np.argmin(S))
            if S[j] < bt:
                bt, bp, bj = S[j], p, j
        chi = per_gal(tab, lamf(bp), "dat")[:, bj].sum()
        K = len(NAMES) + 1 + k
        best[m] = dict(p=bp, j=bj, tot=bt)
        rows.append(dict(Modelo=m, parametros_lambda=k, chi2=round(chi), BIC=round(chi + K * np.log(npts)),
                         g_dagger=f"{GD_GRID[bj]*TO_MS2:.3g}",
                         mejor=", ".join(f"{a}={b:.3g}" for a, b in bp.items()) or "–"))
    d = pd.DataFrame(rows).set_index("Modelo"); d["ΔBIC"] = d.BIC - d.BIC.min()
    return d, best

RESULTS = {}
for label, tab in TABLES.items():
    t0 = time.time(); RESULTS[label] = compete(tab)
    print(f"\n=== {label} ({time.time()-t0:.0f} s) ===  [θ en segundos de arco, L y Lf en kpc]")
    print(RESULTS[label][0].to_string())

## 6. ¿Queda una λ física cuando se descuenta el haz?

Perfil del modelo combinado: para cada λ_f, el mejor θ (efecto del haz) y g†. Si el mínimo está en λ_f = 0, toda la señal la explica el instrumento. Si está en λ_f > 0 y la caída supera la penalización BIC (línea gris), sobrevive una componente física.

In [ ]:
LF = np.r_[0, np.geomspace(0.02, 50, 39)]; THS = np.r_[0, np.geomspace(0.3, 300, 40)]
fig, ax = plt.subplots(figsize=(8, 4.2))
PHYS = {}
for (label, tab), c in zip(TABLES.items(), [COL["RAR no local (ER)"], COL["RAR local (g† global)"], COL["Instrumental"]]):
    prof = np.array([min(per_gal(tab, np.sqrt(TH(t) ** 2 + lf ** 2)).sum(0).min() for t in THS) for lf in LF])
    prof -= prof.min()
    PHYS[label] = dict(lf=LF[np.argmin(prof)], gain=prof[0])
    ax.semilogx(np.r_[0.01, LF[1:]], np.clip(prof, 0.1, None), "o-", ms=3.5, color=c, label=label)
ax.axhline(np.log(N_TOT_PTS), color="#9a9a95", ls="--", lw=1, label="Penalización BIC de 1 parámetro")
ax.set_yscale("log"); ax.set_xticks([0.01, 0.1, 1, 10]); ax.set_xticklabels(["0", "0.1", "1", "10"])
ax.set_xlabel("λ_f física [kpc] (tras descontar el haz)"); ax.set_ylabel("Δχ² respecto al mejor")
ax.set_title("¿Sobrevive una λ física?", loc="left"); ax.legend(fontsize=8.5)
plt.show()
for label, r in PHYS.items():
    print(f"[{label}] mejor λ_f = {r['lf']:.3g} kpc | mejora frente a λ_f = 0: Δχ² = {r['gain']:.1f}")

## 7. Galaxia a galaxia: ¿de qué depende λ?

Para cada galaxia buscamos su λ preferido (con el g† del modelo de λ constante, tabla con 5 % sistemático) y hacemos una **regresión múltiple** de log λ sobre log D, log r_max y log Σ_b a la vez. Así se separan efectos que van juntos: en la muestra, las galaxias grandes también están más lejos.
- Si domina **D** → instrumental.
- Si domina **Σ_b** (con signo negativo: más λ en galaxias difusas) → encaja con la decoherencia.
- Si domina **r_max** → efecto del modelado (geometría del disco).

> **Precaución:** esta regresión es orientativa. En las pruebas con datos simulados, una λ física constante también produce cierta correlación con la distancia, porque las galaxias lejanas tienen menos resolución para medir λ pequeñas. Lo que decide es la competición de modelos de la sección 5 y el perfil de la sección 6.

In [ ]:
tab = TABLES["5 % sistemático"]; jb = RESULTS["5 % sistemático"][1]["λ constante"]["j"]
c = tab["tot"][:, :, jb]
LAMI = LAM_GRID[np.argmin(c, 1)]; GAIN = c[:, 0] - c.min(1)
sel = (GAIN > 4) & (LAMI > 0)
print(f"Galaxias con preferencia clara por λ > 0: {sel.sum()} de {len(NAMES)}")

X = np.c_[np.log10(D), np.log10(RMAX), np.log10(SIGMA)][sel]
y = np.log10(LAMI[sel])
Xs = (X - X.mean(0)) / X.std(0)
A = np.c_[np.ones(sel.sum()), Xs]
coef, *_ = np.linalg.lstsq(A, y, rcond=None)
res = y - A @ coef
cov = np.linalg.inv(A.T @ A) * (res @ res) / max(sel.sum() - 4, 1)
se = np.sqrt(np.diag(cov))
REG = pd.DataFrame(dict(coef=coef[1:], err=se[1:], t=coef[1:] / se[1:]), index=["log D", "log r_max", "log Σ_b"]).round(3)
print("Regresión múltiple de log λ (variables estandarizadas; |t| > 3 = efecto claro):")
print(REG.to_string())
for nm, col in zip(["D", "r_max", "Σ_b"], X.T):
    rr, pp = spearmanr(col, y); print(f"  Spearman log λ – log {nm}: ρ = {rr:.2f} (p = {pp:.0e})")

theta_i = LAMI[sel] / (D[sel] * 1e3) * ARCSEC
print(f"\nλ en ángulo: mediana {np.median(theta_i):.1f}″ (16–84 %: {np.percentile(theta_i,16):.1f}–{np.percentile(theta_i,84):.1f}″); "
      f"dispersión {np.std(np.log10(theta_i)):.2f} dex en ángulo frente a {np.std(y):.2f} dex en kpc")

fig, axs = plt.subplots(1, 3, figsize=(13, 3.9), sharey=True)
for ax, xv, lab in zip(axs, [D, RMAX, SIGMA], ["Distancia [Mpc]", "Radio máximo [kpc]", "Σ_b [M☉/pc²]"]):
    ax.loglog(xv[~sel], np.where(LAMI[~sel] == 0, 0.015, LAMI[~sel]), "o", ms=3.5, color="#c3c2b7", label="Sin preferencia clara")
    ax.loglog(xv[sel], LAMI[sel], "o", ms=4.5, color=COL["RAR no local (ER)"], label="Prefiere λ > 0")
    ax.set_xlabel(lab)
axs[0].set_ylabel("λ preferido [kpc]")
dd = np.geomspace(D.min(), D.max(), 20)
for th, ls in [(10, ":"), (30, "--")]:
    axs[0].loglog(dd, th / ARCSEC * dd * 1e3, ls, color=COL["Instrumental"], lw=1.5, label=f"Haz de {th}″")
axs[0].legend(fontsize=8); axs[0].set_title("λ frente a distancia, tamaño y densidad", loc="left")
plt.tight_layout(); plt.show()

## 8. Comparación con NFW en las mismas condiciones

In [ ]:
def _v2_nfw_(r, V200, c):
    x = r / (V200 / (10 * H0_KPC)); mu = lambda y: np.log1p(y) - y / (1 + y)
    return V200**2 * mu(c * x) / (x * mu(c))
def c_dutton(V200):
    M200 = V200**3 / (10 * G * H0_KPC)
    return 10 ** (0.905 - 0.101 * np.log10(M200 / (1e12 / (H0 / 100))))

def fit_nfw(g, Dmpc, cm, floor, cut):
    err = np.sqrt(g["err"] ** 2 + (floor * g["vobs"]) ** 2)
    mask = np.ones(len(g["r"]), bool)
    if cut:
        mask = g["r"] >= cut * BEAM_ARCSEC / ARCSEC * Dmpc * 1e3
        if mask.sum() < 5: mask = np.zeros(len(g["r"]), bool); mask[-5:] = True
    if cm:
        f = lambda p: vb2(g, p[0]) + _v2_nfw_(g["r"], p[1], c_dutton(p[1])); lo, hi, p0 = [0.1, 10], [1.5, 500], [0.5, 100]
    else:
        f = lambda p: vb2(g, p[0]) + _v2_nfw_(g["r"], p[1], p[2]); lo, hi, p0 = [0.1, 10, 1], [1.5, 500, 50], [0.5, 100, 10]
    best = None
    for st in [p0] + [list(np.array(lo) + RNG.random(len(lo)) * (np.array(hi) - np.array(lo))) for _ in range(3)]:
        r = least_squares(lambda p: np.r_[((g["vobs"] - np.sqrt(np.clip(f(p), 0, None))) / err)[mask], prior(p[0])], st, bounds=(lo, hi))
        best = r if best is None or r.cost < best.cost else best
    v = np.sqrt(np.clip(f(best.x), 0, None))
    return float(((((g["vobs"] - v) / err) ** 2)[mask]).sum()), len(best.x)

rows = []
t0 = time.time()
for label, kw in [("Errores originales", (0.0, None)), ("5 % sistemático", (0.05, None)), ("5 % + sin interior", (0.05, 2))]:
    npts = TABLES[label]["npts"].sum(); d = RESULTS[label][0]
    for m in ["RAR local", "λ constante", "Instrumental (θ·D)", "Instrumental + físico"]:
        rows.append(dict(condiciones=label, Modelo=m, BIC=d.loc[m, "BIC"]))
    for m, cm in [("NFW ΛCDM (c–M)", True), ("NFW libre", False)]:
        fits = [fit_nfw(GAL[n], D[i], cm, *kw) for i, n in enumerate(NAMES)]
        rows.append(dict(condiciones=label, Modelo=m, BIC=round(sum(f[0] for f in fits) + sum(f[1] for f in fits) * np.log(npts))))
CMP = pd.DataFrame(rows)
CMP["ΔBIC"] = CMP.groupby("condiciones").BIC.transform(lambda b: b - b.min())
print(f"({time.time()-t0:.0f} s)")
CMP.pivot(index="Modelo", columns="condiciones", values="ΔBIC")

## 9. Veredicto automático

In [ ]:
def veredicto():
    if not HAVE_SPARC:
        print("⚠️  Datos SINTÉTICOS: solo comprueba que el código funciona.\n")
    for label in TABLES:
        d = RESULTS[label][0]; b = d.BIC
        print(f"[{label}]")
        print(f"   Instrumental frente a λ constante: ΔBIC = {b['λ constante'] - b['Instrumental (θ·D)']:+.0f} (positivo = gana instrumental)")
        print(f"   Densidad frente a λ constante:     ΔBIC = {b['λ constante'] - b['Densidad λ(Σ)']:+.0f} (positivo = gana densidad)")
        print(f"   Componente física tras el haz: λ_f = {PHYS[label]['lf']:.3g} kpc, Δχ² = {PHYS[label]['gain']:.1f}")
        print(f"   Mejor modelo: {d.BIC.idxmin()}")
    s5, cut = RESULTS["5 % sistemático"][0].BIC, RESULTS["5 % + sin interior"][0].BIC
    instr = (s5["λ constante"] - s5["Instrumental (θ·D)"] > 10) and (cut["λ constante"] - cut["Instrumental (θ·D)"] > -10)
    phys = all(PHYS[l]["gain"] > 10 and PHYS[l]["lf"] > 0 for l in ["5 % sistemático", "5 % + sin interior"])
    dens = (s5["λ constante"] - s5["Densidad λ(Σ)"] > 10) and (s5["Instrumental (θ·D)"] - s5["Densidad λ(Σ)"] > 10)
    tD, tS = REG.loc["log D", "t"], REG.loc["log Σ_b", "t"]
    print(f"\nRegresión galaxia a galaxia: t(distancia) = {tD:+.1f}, t(densidad) = {tS:+.1f}")
    print()
    if s5.idxmin() == "RAR local" and cut.idxmin() == "RAR local":
        print("→ SIN SEÑAL: con errores realistas, la RAR local (λ = 0) es el mejor modelo. No hay no localidad que explicar.")
    elif phys:
        print(f"→ SOBREVIVE UNA COMPONENTE FÍSICA: tras descontar el haz queda λ_f ≈ {PHYS['5 % sistemático']['lf']:.2g} kpc, "
              "también sin los puntos interiores. Es la señal candidata de la red ER; hay que confirmarla con datos de mayor resolución.")
    elif instr:
        print("→ LA SEÑAL ES INSTRUMENTAL: λ se explica por un ángulo fijo (el haz del telescopio) y no queda componente física significativa.")
    else:
        print("→ NO CONCLUYENTE: ni el modelo instrumental ni una λ física explican la señal de forma clara.")
    if dens:
        print("→ Además, λ depende de la densidad mejor que de un valor constante o del haz: "
              + ("más alcance en galaxias difusas, como predice la decoherencia." if RESULTS['5 % sistemático'][1]['Densidad λ(Σ)']['p']['gm'] < 0
                 else "pero con más alcance en galaxias DENSAS, al revés de lo que predice la decoherencia."))

veredicto()

## 10. Cómo leer el resultado

- **Si la señal es instrumental:** es un resultado útil y honesto. El notebook 4 encontró algo real en los datos, pero es el difuminado del haz, no los puentes ER. En galaxias, la red ER sigue siendo indistinguible de la RAR local, y la prueba de la hipótesis pasa a cúmulos, lentes y cosmología.
- **Si sobrevive una λ física:** es una candidata a señal nueva. El siguiente paso sería repetirlo con curvas de rotación de alta resolución (por ejemplo, datos de Hα o de interferómetros con haces de pocos segundos de arco), donde el difuminado es mucho menor. Si λ_f se mantiene, es una predicción propia de la red ER.
- **Si λ depende de la densidad** (más alcance en galaxias difusas): encaja con la idea de decoherencia y sería una segunda predicción comprobable. Pero solo cuenta si gana también al modelo instrumental.
- **Limitación:** el haz real cambia de una galaxia a otra (distintos telescopios). Aquí se supone un ángulo común. Un análisis definitivo usaría el haz de cada observación.